# Week 2 Assignment — HOG Feature Engineering + Classifier Comparison
Run top to bottom. If your Colab runtime is fresh, upload `week2_hog_data.zip` when prompted.

In [ ]:
!pip install -q scikit-image scikit-learn opencv-python-headless

In [ ]:
import os

if not os.path.isdir('dataset'):
    from google.colab import files
    uploaded = files.upload()  # choose week2_hog_data.zip
    import zipfile
    with zipfile.ZipFile('week2_hog_data.zip') as z:
        z.extractall('.')

print('dataset classes:', os.listdir('dataset'))
for c in os.listdir('dataset'):
    print(' ', c, '->', len(os.listdir(os.path.join('dataset', c))), 'images')

## Task 2 — Extract HOG Features with Two Parameter Settings

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import exposure

DATASET_DIR = 'dataset'
IMG_SIZE = (128, 128)

SETTINGS = {
    'A': dict(orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2)),
    'B': dict(orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2)),
}

def extract_hog_features(image_path, params):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    img = cv2.resize(img, IMG_SIZE)
    return hog(img, block_norm='L2-Hys', **params)

def build_dataset(params):
    X, y = [], []
    class_names = sorted(os.listdir(DATASET_DIR))
    for label_idx, class_name in enumerate(class_names):
        class_folder = os.path.join(DATASET_DIR, class_name)
        for filename in os.listdir(class_folder):
            path = os.path.join(class_folder, filename)
            try:
                X.append(extract_hog_features(path, params))
                y.append(label_idx)
            except Exception as e:
                print(f'Skipping {path}: {e}')
    return np.array(X), np.array(y), class_names

def save_visualization(params, class_names, setting_name):
    fig, axes = plt.subplots(2, len(class_names), figsize=(4 * len(class_names), 8))
    for i, class_name in enumerate(class_names):
        class_folder = os.path.join(DATASET_DIR, class_name)
        filename = sorted(os.listdir(class_folder))[0]
        path = os.path.join(class_folder, filename)
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = cv2.resize(img, IMG_SIZE)
        _, hog_image = hog(img, block_norm='L2-Hys', visualize=True, **params)
        hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))
        axes[0, i].imshow(img, cmap='gray'); axes[0, i].set_title(f'{class_name} - input'); axes[0, i].axis('off')
        axes[1, i].imshow(hog_image_rescaled, cmap='gray'); axes[1, i].set_title(f'{class_name} - HOG'); axes[1, i].axis('off')
    plt.tight_layout()
    plt.savefig(f'hog_visualization_setting_{setting_name}.png', dpi=150)
    plt.show()

class_names = None
for setting_name, params in SETTINGS.items():
    X, y, class_names = build_dataset(params)
    print(f'Setting {setting_name} {params}: feature vector length = {X.shape[1]}')
    np.save(f'features_setting_{setting_name}.npy', X)
    np.save(f'labels_setting_{setting_name}.npy', y)
    if setting_name == 'A':
        save_visualization(params, class_names, setting_name)

np.save('class_names.npy', np.array(class_names))
print('Classes:', class_names)

## Task 3 — Train and Compare Classifiers (Linear SVM vs k-NN vs Random Forest, Setting A features)

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

X = np.load('features_setting_A.npy')
y = np.load('labels_setting_A.npy')
class_names = np.load('class_names.npy', allow_pickle=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

models = {
    'Linear SVM': LinearSVC(max_iter=10000),
    'k-NN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(f'--- {name} ---')
    print('Accuracy:', acc)
    print('Classification Report:')
    print(classification_report(y_test, preds, target_names=class_names))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, preds))
    print()

print('Summary:')
for name, acc in results.items():
    print(f'{name}: {acc:.3f}')